# LangGraph Vendor Quote Analysis Workflow

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that analyzes vendor quotes, flags commercial and compliance risks, drafts an executive summary, and provides approval decision support — with each state transition explicitly documented.

**Why this matters:** Procurement teams process hundreds of vendor quotes per quarter. Each quote needs cost analysis, risk review, and a recommendation before budget approval. Doing this manually is slow and inconsistent. A graph-based pipeline standardizes the evaluation while keeping humans in the decision loop.

**Pipeline Nodes:**

```
  START
    │
    │  STATE: {quote_data, ...empty fields...}
    ▼
  ┌─────────────────────┐
  │  analyze_quote       │  parse line items, compute totals, benchmark
  └──────────┬──────────┘
             │  STATE += {cost_analysis}
             ▼
  ┌─────────────────────┐
  │  flag_risks          │  identify commercial, compliance, delivery risks
  └──────────┬──────────┘
             │  STATE += {risk_report}
             ▼
  ┌─────────────────────┐
  │  draft_summary       │  write executive summary with recommendation
  └──────────┬──────────┘
             │  STATE += {exec_summary}
             ▼
  ┌─────────────────────┐
  │  approval_decision   │  score and route: approve / negotiate / reject
  └──────────┬──────────┘
        conditional
      ┌────┼──────────┐
      ▼    ▼          ▼
   approve negotiate reject
      │    │          │
      │    └→ draft_summary (with negotiation points)
      │               │
      ▼               ▼
  ┌─────────────────────┐
  │  compile_report      │  assemble final procurement report
  └──────────┬──────────┘
             │  STATE += {final_report}
             ▼
           END
```

**Stack:** `LangGraph`, `ChatOllama` + `qwen3.5:9b` (local, no API keys), `pandas`

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** for a procurement pipeline |
| 2 | Build **single-responsibility nodes** for each evaluation phase |
| 3 | Understand every **state transition** — what enters, what exits, why |
| 4 | Implement **conditional routing** with three decision branches |
| 5 | Add a **negotiation loop** that revises the summary with counter-points |
| 6 | Combine **deterministic analysis** (pandas) with **LLM reasoning** |
| 7 | Evaluate the pipeline across multiple vendor quotes |

## 3. State Transitions — The Core Concept

Every edge in the graph is a **state transition**. Each transition has three properties:

```
  ┌────────────────────────────────────────────────────────────────┐
  │  ANATOMY OF A STATE TRANSITION                                │
  ├────────────────────────────────────────────────────────────────┤
  │                                                                │
  │  1. PRECONDITION — what must be in state before this node     │
  │     Example: flag_risks needs cost_analysis to already exist  │
  │                                                                │
  │  2. TRANSFORMATION — what the node computes                   │
  │     Example: flag_risks inspects costs and flags issues       │
  │                                                                │
  │  3. POSTCONDITION — what the node guarantees in state         │
  │     Example: risk_report is now populated (never empty)       │
  │                                                                │
  │  WHY THIS MATTERS:                                            │
  │  If a node fails, you know EXACTLY which precondition was     │
  │  missing or which postcondition wasn't met.                    │
  └────────────────────────────────────────────────────────────────┘
```

### Transition Map for This Pipeline

```
  TRANSITION               PRECONDITION          POSTCONDITION
  ──────────────────────── ───────────────────── ─────────────────────
  START → analyze_quote    quote_data exists     cost_analysis filled
  analyze → flag_risks     cost_analysis exists  risk_report filled
  flag → draft_summary     risk_report exists    exec_summary filled
  draft → approval         exec_summary exists   decision + feedback
  approval → compile       decision is final     final_report filled
  approval → draft (loop)  decision="negotiate"  feedback for revision
```

## 4. Why Partial State Updates?

```
  After analyze_quote:
    state = {quote_data: {...}, cost_analysis: "...",
             risk_report: "", exec_summary: "", ...}
                                ↑ empty — not yet written

  After flag_risks:
    state = {quote_data: {...}, cost_analysis: "...",
             risk_report: "3 risks found...", exec_summary: "", ...}
    ↑ cost_analysis preserved       ↑ now filled

  RULE: Each node returns ONLY what it changed.
  LangGraph merges the partial update into the full state.
  Nothing gets accidentally overwritten.
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph pandas numpy

print("Dependencies: langchain, langgraph, pandas")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict

import numpy as np
import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Vendor Quotes & State Schema

## 8. Simulated Vendor Quotes

Three quotes for a cloud infrastructure contract, each with different pricing structures, terms, and risk profiles.

In [ ]:
VENDOR_QUOTES = [
    {
        "vendor_name": "CloudPeak Infrastructure",
        "quote_id": "CPK-2024-0891",
        "submitted": "2024-11-10",
        "valid_until": "2024-12-10",
        "contract_term": "36 months",
        "line_items": [
            {"item": "Compute (Reserved Instances)", "qty": 50, "unit": "instances/mo", "unit_price": 420, "annual": 252000},
            {"item": "Block Storage (SSD)", "qty": 100, "unit": "TB/mo", "unit_price": 85, "annual": 102000},
            {"item": "Network Egress", "qty": 200, "unit": "TB/mo", "unit_price": 45, "annual": 108000},
            {"item": "Managed Kubernetes", "qty": 5, "unit": "clusters/mo", "unit_price": 1200, "annual": 72000},
            {"item": "24/7 Premium Support", "qty": 1, "unit": "contract", "unit_price": 4500, "annual": 54000},
            {"item": "Security & Compliance Add-on", "qty": 1, "unit": "contract", "unit_price": 2800, "annual": 33600},
        ],
        "total_annual": 621600,
        "total_contract": 1864800,
        "payment_terms": "Net 30, annual prepay with 5% discount",
        "sla": "99.95% uptime, 15-min P1 response, 4-hour resolution target",
        "termination_clause": "90-day notice, no early termination fee after month 18",
        "data_residency": "US-East and US-West regions only",
        "certifications": ["SOC 2 Type II", "ISO 27001", "HIPAA BAA available"],
        "notes": "Volume discount of 8% if compute exceeds 75 instances.",
    },
    {
        "vendor_name": "NimbusScale",
        "quote_id": "NS-Q-4420",
        "submitted": "2024-11-12",
        "valid_until": "2024-11-30",
        "contract_term": "24 months",
        "line_items": [
            {"item": "Compute (On-Demand)", "qty": 50, "unit": "instances/mo", "unit_price": 580, "annual": 348000},
            {"item": "Block Storage (SSD)", "qty": 100, "unit": "TB/mo", "unit_price": 72, "annual": 86400},
            {"item": "Network Egress", "qty": 200, "unit": "TB/mo", "unit_price": 38, "annual": 91200},
            {"item": "Managed Kubernetes", "qty": 5, "unit": "clusters/mo", "unit_price": 950, "annual": 57000},
            {"item": "Business Support", "qty": 1, "unit": "contract", "unit_price": 2500, "annual": 30000},
        ],
        "total_annual": 612600,
        "total_contract": 1225200,
        "payment_terms": "Net 45, quarterly billing",
        "sla": "99.9% uptime, 30-min P1 response, 8-hour resolution target",
        "termination_clause": "60-day notice, 3-month early termination penalty",
        "data_residency": "Multi-region (US, EU, APAC)",
        "certifications": ["SOC 2 Type II", "ISO 27001"],
        "notes": "On-demand pricing — no reserved commitment. Quote expires in 18 days.",
    },
    {
        "vendor_name": "TerraVault Cloud",
        "quote_id": "TV-2024-3301",
        "submitted": "2024-11-14",
        "valid_until": "2025-01-14",
        "contract_term": "36 months",
        "line_items": [
            {"item": "Compute (Reserved)", "qty": 50, "unit": "instances/mo", "unit_price": 395, "annual": 237000},
            {"item": "Object + Block Storage", "qty": 150, "unit": "TB/mo", "unit_price": 60, "annual": 108000},
            {"item": "Network Egress", "qty": 200, "unit": "TB/mo", "unit_price": 52, "annual": 124800},
            {"item": "Managed Kubernetes", "qty": 5, "unit": "clusters/mo", "unit_price": 1100, "annual": 66000},
            {"item": "Enterprise Support", "qty": 1, "unit": "contract", "unit_price": 6000, "annual": 72000},
            {"item": "Compliance Suite (SOC/HIPAA/PCI)", "qty": 1, "unit": "contract", "unit_price": 3500, "annual": 42000},
            {"item": "Data Migration Services", "qty": 1, "unit": "one-time", "unit_price": 45000, "annual": 15000},
        ],
        "total_annual": 664800,
        "total_contract": 1994400,
        "payment_terms": "Net 30, monthly billing, no prepay discount",
        "sla": "99.99% uptime, 10-min P1 response, 2-hour resolution target",
        "termination_clause": "180-day notice, 6-month early termination penalty",
        "data_residency": "US-only with FedRAMP Moderate authorization",
        "certifications": ["SOC 2 Type II", "ISO 27001", "HIPAA BAA", "FedRAMP Moderate", "PCI DSS"],
        "notes": "Includes free data migration (valued at $45K). Highest SLA tier.",
    },
]

# Internal benchmark data for comparison
BENCHMARK = {
    "compute_per_instance_mo": {"low": 350, "market": 450, "high": 600},
    "storage_per_tb_mo": {"low": 55, "market": 80, "high": 110},
    "egress_per_tb_mo": {"low": 30, "market": 45, "high": 65},
    "kubernetes_per_cluster_mo": {"low": 800, "market": 1100, "high": 1500},
    "support_annual": {"low": 24000, "market": 48000, "high": 72000},
}

print(f"Vendor quotes: {len(VENDOR_QUOTES)}")
for q in VENDOR_QUOTES:
    print(f"  {q['vendor_name']} ({q['quote_id']}): ${q['total_annual']:,}/yr, "
          f"{q['contract_term']}, {len(q['line_items'])} line items")

## 9. State Schema

**Field ownership — annotated with state transitions:**

| Field | Written By | Read By | Transition |
|-------|-----------|--------|------------|
| `quote_data` | Input | analyze_quote | START → analyze |
| `benchmark` | Input | analyze_quote | START → analyze |
| `cost_analysis` | analyze_quote | flag_risks, draft_summary | analyze → flag |
| `risk_report` | flag_risks | draft_summary, approval_decision | flag → draft |
| `exec_summary` | draft_summary | approval_decision | draft → approval |
| `decision` | approval_decision | routing function | approval → route |
| `decision_feedback` | approval_decision | draft_summary (on negotiate) | approval → draft (loop) |
| `revision_count` | approval_decision | routing function | approval → route |
| `final_report` | compile_report | caller | compile → END |

In [ ]:
class QuoteState(TypedDict):
    """State schema for the vendor quote analysis pipeline.

    Design notes:
    - quote_data: the vendor's raw quote (immutable after init)
    - benchmark: internal pricing benchmarks (immutable after init)
    - cost_analysis: line-item breakdown with benchmark comparison
    - risk_report: flagged risks with severity and mitigation
    - exec_summary: executive-ready summary with recommendation
    - decision: approve / negotiate / reject
    - decision_feedback: negotiation points or rejection reasons
    - revision_count: caps the negotiate loop
    - final_report: compiled procurement report
    - current_node: debugging
    """
    quote_data: dict
    benchmark: dict
    cost_analysis: str
    risk_report: str
    exec_summary: str
    decision: str
    decision_feedback: str
    revision_count: int
    final_report: str
    current_node: str


print("State schema: QuoteState")
for name, typ in QuoteState.__annotations__.items():
    print(f"  {name}: {typ}")

---

# Part B — Build the Graph Nodes

## 10. Node 1: Analyze Quote

**Transition: START → analyze_quote**

```
  PRECONDITION:  quote_data exists with line_items and totals
                 benchmark exists with market pricing
  TRANSFORM:     compute per-item cost vs benchmark, flag above/below market
  POSTCONDITION: cost_analysis is a non-empty string with benchmarked breakdown
```

This node is **hybrid** — it uses pandas for deterministic math and the LLM for narrative interpretation.

In [ ]:
def analyze_quote(state: QuoteState) -> dict:
    """Node: Parse line items and benchmark against market pricing.

    Transition: START → analyze_quote
    Precondition: quote_data and benchmark in state
    Postcondition: cost_analysis populated with benchmarked breakdown
    """
    print("  [NODE] analyze_quote")
    q = state["quote_data"]
    bench = state["benchmark"]

    # Deterministic: build cost comparison table
    lines = []
    lines.append(f"COST ANALYSIS — {q['vendor_name']} ({q['quote_id']})")
    lines.append(f"Contract: {q['contract_term']}, Total: ${q['total_contract']:,}")
    lines.append(f"Annual: ${q['total_annual']:,}")
    lines.append("")
    lines.append(f"{'Line Item':<30} {'Unit Price':>12} {'Market':>10} {'Δ':>10} {'Status'}")
    lines.append("-" * 80)

    # Map line items to benchmark categories
    category_map = {
        "compute": "compute_per_instance_mo",
        "storage": "storage_per_tb_mo",
        "block": "storage_per_tb_mo",
        "object": "storage_per_tb_mo",
        "network": "egress_per_tb_mo",
        "egress": "egress_per_tb_mo",
        "kubernetes": "kubernetes_per_cluster_mo",
        "support": "support_annual",
    }

    total_savings = 0
    for item in q["line_items"]:
        name_lower = item["item"].lower()
        bench_key = None
        for keyword, key in category_map.items():
            if keyword in name_lower:
                bench_key = key
                break

        if bench_key and bench_key in bench:
            market = bench[bench_key]["market"]
            is_annual = "support" in name_lower
            price = item["annual"] if is_annual else item["unit_price"]
            compare = market
            delta = price - compare
            pct = round(100 * delta / compare, 1) if compare else 0
            status = "✓ BELOW" if delta < 0 else ("~ MARKET" if abs(pct) < 10 else "⚠ ABOVE")
            total_savings += delta * (1 if is_annual else item["qty"] * 12)
            lines.append(f"  {item['item']:<28} ${price:>10,} ${compare:>8,} {pct:>+8.1f}%  {status}")
        else:
            lines.append(f"  {item['item']:<28} ${item['unit_price']:>10,}  {'—':>8}  {'—':>8}  (no benchmark)")

    lines.append("")
    lines.append(f"Estimated annual delta vs market: ${total_savings:+,.0f}")
    lines.append(f"Payment terms: {q['payment_terms']}")
    lines.append(f"SLA: {q['sla']}")

    analysis = "\n".join(lines)
    print(f"    Analysis: {len(analysis)} chars, {len(q['line_items'])} line items benchmarked")
    return {"cost_analysis": analysis, "current_node": "analyze_quote"}


print("Node defined: analyze_quote")
print("  READS:  quote_data, benchmark")
print("  WRITES: cost_analysis")
print("  TYPE:   deterministic (pandas math, no LLM)")

## 11. Node 2: Flag Risks

**Transition: analyze_quote → flag_risks**

```
  PRECONDITION:  cost_analysis exists (from analyze_quote)
                 quote_data has SLA, termination, certification fields
  TRANSFORM:     LLM inspects costs, terms, and compliance for risks
  POSTCONDITION: risk_report with categorized, severity-rated risks
```

In [ ]:
RISK_SYSTEM = """You are a procurement risk analyst. Given a vendor quote and cost analysis,
identify risks in these categories:

1. COMMERCIAL RISKS — pricing above market, unfavorable payment terms, hidden costs
2. COMPLIANCE RISKS — missing certifications, data residency concerns, audit gaps
3. DELIVERY RISKS — weak SLA, long resolution times, limited support coverage
4. CONTRACT RISKS — lock-in, termination penalties, auto-renewal traps
5. VENDOR RISKS — quote expiry pressure, missing guarantees

For each risk:
- SEVERITY: critical / high / medium / low
- DESCRIPTION: what the risk is
- MITIGATION: how to address it in negotiation

Be specific — cite exact numbers and clauses from the quote."""


def flag_risks(state: QuoteState) -> dict:
    """Node: Identify and categorize risks in the quote.

    Transition: analyze_quote → flag_risks
    Precondition: cost_analysis exists
    Postcondition: risk_report populated
    """
    print("  [NODE] flag_risks")
    q = state["quote_data"]

    report = ask(
        f"VENDOR: {q['vendor_name']}\n\n"
        f"COST ANALYSIS:\n{state['cost_analysis'][:2000]}\n\n"
        f"CONTRACT TERMS:\n"
        f"  Term: {q['contract_term']}\n"
        f"  SLA: {q['sla']}\n"
        f"  Termination: {q['termination_clause']}\n"
        f"  Data Residency: {q['data_residency']}\n"
        f"  Certifications: {', '.join(q['certifications'])}\n"
        f"  Payment: {q['payment_terms']}\n"
        f"  Quote Valid Until: {q['valid_until']}\n"
        f"  Notes: {q.get('notes', 'None')}\n\n"
        f"Identify all risks with severity, description, and mitigation.",
        system=RISK_SYSTEM,
        temperature=0.1,
    )
    print(f"    Risk report: {len(report)} chars")
    return {"risk_report": report, "current_node": "flag_risks"}


print("Node defined: flag_risks")
print("  READS:  quote_data, cost_analysis")
print("  WRITES: risk_report")

## 12. Node 3: Draft Summary

**Transition: flag_risks → draft_summary**

```
  PRECONDITION:  cost_analysis AND risk_report both exist
  TRANSFORM:     LLM synthesizes costs + risks into executive summary
  POSTCONDITION: exec_summary with recommendation (concise, decision-ready)
```

If this is a revision due to a "negotiate" decision, the node also incorporates negotiation feedback.

In [ ]:
SUMMARY_SYSTEM = """Write an executive procurement summary for a decision-maker.
Include:
1. VENDOR OVERVIEW (1-2 sentences)
2. COST POSITION (vs market — above, at, or below)
3. TOP RISKS (3 most critical)
4. STRENGTHS (what's good about this quote)
5. RECOMMENDATION (approve / negotiate / reject, with clear rationale)

Write for a CFO — be direct, quantitative, and actionable. 200 words max."""

SUMMARY_PROMPT = """VENDOR: {vendor}

COST ANALYSIS:
{costs}

RISK REPORT:
{risks}

{revision_section}

Write the executive summary with recommendation."""


def draft_summary(state: QuoteState) -> dict:
    """Node: Write an executive summary with recommendation.

    Transition: flag_risks → draft_summary (or approval → draft_summary on negotiate)
    Precondition: cost_analysis and risk_report exist
    Postcondition: exec_summary populated
    """
    print("  [NODE] draft_summary")
    q = state["quote_data"]

    revision_section = ""
    if state.get("decision_feedback") and state.get("revision_count", 0) > 0:
        revision_section = (
            f"NEGOTIATION FEEDBACK (incorporate these points):\n"
            f"{state['decision_feedback']}\n\n"
            f"PREVIOUS SUMMARY (revise, don't restart):\n"
            f"{state['exec_summary'][:500]}"
        )
        print(f"    Revision round {state['revision_count']} — adding negotiation points")

    summary = ask(
        SUMMARY_PROMPT.format(
            vendor=q["vendor_name"],
            costs=state["cost_analysis"][:1500],
            risks=state["risk_report"][:1500],
            revision_section=revision_section,
        ),
        system=SUMMARY_SYSTEM,
        temperature=0.2,
    )
    print(f"    Summary: {len(summary)} chars")
    return {"exec_summary": summary, "current_node": "draft_summary"}


print("Node defined: draft_summary")
print("  READS:  quote_data, cost_analysis, risk_report, decision_feedback (if revision)")
print("  WRITES: exec_summary")

## 13. Node 4: Approval Decision

**Transition: draft_summary → approval_decision**

```
  PRECONDITION:  exec_summary exists (with recommendation)
                 cost_analysis and risk_report available for scoring
  TRANSFORM:     LLM scores on 5 dimensions, routes to a decision
  POSTCONDITION: decision is one of {approve, negotiate, reject}
                 decision_feedback explains rationale/negotiation points
```

### Three-Way Routing

```
  APPROVE   (score ≥ 4.0)  → compile_report — ready for budget sign-off
  NEGOTIATE (score 2.5-3.9) → draft_summary  — revise with counter-points
  REJECT    (score < 2.5)  → compile_report — document why and move on
```

In [ ]:
DECISION_SYSTEM = """You are the CFO reviewing a vendor quote. Score it and decide.
Return valid JSON only."""

DECISION_PROMPT = """EXECUTIVE SUMMARY:
{summary}

COST ANALYSIS:
{costs}

RISK REPORT (first 500 chars):
{risks}

Score each dimension 1-5 and decide.
{{
  "cost_competitiveness": 1-5,
  "risk_profile": 1-5,
  "contract_terms": 1-5,
  "compliance_readiness": 1-5,
  "overall_value": 1-5,
  "average_score": float,
  "decision": "approve" or "negotiate" or "reject",
  "feedback": "rationale and negotiation points if applicable"
}}"""


def approval_decision(state: QuoteState) -> dict:
    """Node: Score the quote and route to a decision.

    Transition: draft_summary → approval_decision
    Precondition: exec_summary, cost_analysis, risk_report exist
    Postcondition: decision in {approve, negotiate, reject}
    """
    print("  [NODE] approval_decision")

    raw = ask(
        DECISION_PROMPT.format(
            summary=state["exec_summary"][:1000],
            costs=state["cost_analysis"][:800],
            risks=state["risk_report"][:500],
        ),
        system=DECISION_SYSTEM,
        temperature=0.1,
    )
    result = parse_json(raw)

    if not result:
        result = {
            "cost_competitiveness": 3, "risk_profile": 3,
            "contract_terms": 3, "compliance_readiness": 3,
            "overall_value": 3, "average_score": 3.0,
            "decision": "negotiate",
            "feedback": "JSON parse fallback — defaulting to negotiate",
        }

    dims = ["cost_competitiveness", "risk_profile", "contract_terms",
            "compliance_readiness", "overall_value"]
    scores = [result.get(d, 3) for d in dims]
    avg = round(sum(scores) / len(scores), 2)
    result["average_score"] = avg

    # Override decision based on score bands
    if avg >= 4.0:
        result["decision"] = "approve"
    elif avg < 2.5:
        result["decision"] = "reject"
    else:
        result["decision"] = "negotiate"

    revision_count = state.get("revision_count", 0)

    print(f"    Scores: {', '.join(f'{d}={result.get(d)}' for d in dims)}")
    print(f"    Average: {result['average_score']} → {result['decision'].upper()}")

    return {
        "decision": result["decision"],
        "decision_feedback": result.get("feedback", ""),
        "revision_count": revision_count + 1,
        "current_node": "approval_decision",
    }


print("Node defined: approval_decision")
print("  READS:  exec_summary, cost_analysis, risk_report")
print("  WRITES: decision, decision_feedback, revision_count")

## 14. Node 5: Compile Report

**Transition: approval_decision → compile_report (terminal)**

```
  PRECONDITION:  ALL prior fields populated
                 decision is final (approve, reject, or negotiate-at-cap)
  TRANSFORM:     assemble all outputs into a formal procurement report
  POSTCONDITION: final_report is a complete, audit-ready document
```

In [ ]:
def compile_report(state: QuoteState) -> dict:
    """Node: Compile the final procurement report.

    Transition: approval_decision → compile_report
    Precondition: all analysis fields populated, decision finalized
    Postcondition: final_report ready for stakeholders
    """
    print("  [NODE] compile_report")
    q = state["quote_data"]

    status_map = {
        "approve": "✓ APPROVED — Ready for budget sign-off",
        "negotiate": "↔ NEGOTIATE — Proceed with counter-proposal",
        "reject": "✗ REJECTED — Do not proceed",
    }
    status = status_map.get(state["decision"], state["decision"].upper())

    report = (
        f"{'=' * 65}\n"
        f"PROCUREMENT REPORT\n"
        f"{'=' * 65}\n"
        f"Vendor:     {q['vendor_name']}\n"
        f"Quote ID:   {q['quote_id']}\n"
        f"Term:       {q['contract_term']}\n"
        f"Annual:     ${q['total_annual']:,}\n"
        f"Total:      ${q['total_contract']:,}\n"
        f"Decision:   {status}\n"
        f"Revisions:  {state.get('revision_count', 0)}\n"
        f"{'=' * 65}\n\n"
        f"--- COST ANALYSIS ---\n"
        f"{state['cost_analysis']}\n\n"
        f"--- RISK REPORT ---\n"
        f"{state['risk_report']}\n\n"
        f"--- EXECUTIVE SUMMARY ---\n"
        f"{state['exec_summary']}\n\n"
        f"--- DECISION RATIONALE ---\n"
        f"{state['decision_feedback']}\n"
    )

    print(f"    Report: {len(report)} chars")
    return {"final_report": report, "current_node": "compile_report"}


print("Node defined: compile_report")
print("  READS:  ALL state fields")
print("  WRITES: final_report")

---

# Part C — Routing & Graph Assembly

## 15. Routing After Approval Decision

**Three-way conditional edge — explained:**

```
  ┌──────────────────────────────────────────────────────┐
  │            ROUTING LOGIC                              │
  ├──────────────────────────────────────────────────────┤
  │                                                      │
  │  IF decision == "approve":                           │
  │    TRANSITION: approval → compile_report             │
  │    REASON: score ≥ 4.0, quote is competitive         │
  │                                                      │
  │  IF decision == "reject":                            │
  │    TRANSITION: approval → compile_report             │
  │    REASON: score < 2.5, fundamental problems         │
  │                                                      │
  │  IF decision == "negotiate" AND under revision cap:  │
  │    TRANSITION: approval → draft_summary              │
  │    REASON: score 2.5-3.9, worth negotiating          │
  │    The summary gets revised with negotiation points   │
  │                                                      │
  │  IF decision == "negotiate" AND AT revision cap:     │
  │    TRANSITION: approval → compile_report             │
  │    REASON: max negotiation rounds reached             │
  └──────────────────────────────────────────────────────┘
```

In [ ]:
MAX_REVISIONS = 2


def route_after_decision(state: QuoteState) -> str:
    """Conditional edge: three-way routing after approval decision."""
    decision = state.get("decision", "negotiate")
    revisions = state.get("revision_count", 0)

    if decision == "approve":
        print(f"    [ROUTE] approved (score ≥ 4.0) → compile_report")
        return "compile_report"

    if decision == "reject":
        print(f"    [ROUTE] rejected (score < 2.5) → compile_report")
        return "compile_report"

    # decision == "negotiate"
    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] negotiate at cap ({MAX_REVISIONS}) → compile_report")
        return "compile_report"

    print(f"    [ROUTE] negotiate (round {revisions}) → draft_summary")
    return "draft_summary"


print(f"Routing: route_after_decision (max {MAX_REVISIONS} negotiation rounds)")
print("  approve              → compile_report")
print("  reject               → compile_report")
print("  negotiate (under cap) → draft_summary")
print("  negotiate (at cap)   → compile_report")

## 16. Assemble the StateGraph

Each `add_edge` maps to a state transition from the table above.

In [ ]:
workflow = StateGraph(QuoteState)

# Register nodes
workflow.add_node("analyze_quote", analyze_quote)
workflow.add_node("flag_risks", flag_risks)
workflow.add_node("draft_summary", draft_summary)
workflow.add_node("approval_decision", approval_decision)
workflow.add_node("compile_report", compile_report)

# Transition 1: START → analyze_quote
#   Precondition: quote_data in initial state
workflow.add_edge(START, "analyze_quote")

# Transition 2: analyze_quote → flag_risks
#   Precondition: cost_analysis now exists
workflow.add_edge("analyze_quote", "flag_risks")

# Transition 3: flag_risks → draft_summary
#   Precondition: risk_report now exists
workflow.add_edge("flag_risks", "draft_summary")

# Transition 4: draft_summary → approval_decision
#   Precondition: exec_summary now exists
workflow.add_edge("draft_summary", "approval_decision")

# Transition 5: conditional — three-way split
workflow.add_conditional_edges(
    "approval_decision",
    route_after_decision,
    {
        "compile_report": "compile_report",
        "draft_summary": "draft_summary",
    },
)

# Transition 6: compile_report → END
#   Postcondition: final_report populated
workflow.add_edge("compile_report", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("State transitions:")
print("  T1: START        → analyze_quote     (parse costs)")
print("  T2: analyze      → flag_risks        (identify risks)")
print("  T3: flag_risks   → draft_summary     (write summary)")
print("  T4: draft_summary→ approval_decision (score + decide)")
print("  T5: approval     → compile_report    (approve/reject)")
print("      approval     → draft_summary     (negotiate loop)")
print("  T6: compile      → END")

In [ ]:
# Visualize graph
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> analyze_quote --> flag_risks --> draft_summary")
    print("  draft_summary --> approval_decision")
    print("  approval_decision --(approve/reject)--> compile_report --> __end__")
    print("  approval_decision --(negotiate)--> draft_summary")

---

# Part D — Execute the Pipeline

## 17. Run on Quote 1 — CloudPeak Infrastructure

In [ ]:
def make_initial_state(quote: dict) -> QuoteState:
    return {
        "quote_data": quote,
        "benchmark": BENCHMARK,
        "cost_analysis": "",
        "risk_report": "",
        "exec_summary": "",
        "decision": "",
        "decision_feedback": "",
        "revision_count": 0,
        "final_report": "",
        "current_node": "start",
    }


q1 = VENDOR_QUOTES[0]
print(f"Pipeline: {q1['vendor_name']} — ${q1['total_annual']:,}/yr")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(q1),
    {"recursion_limit": 20},
)

print(f"\nComplete.")
print(f"  Decision: {result_1['decision'].upper()}")
print(f"  Revisions: {result_1['revision_count']}")
print(f"  Report: {len(result_1['final_report'])} chars")

## 18. Inspect Intermediate State — Transition by Transition

In [ ]:
print("STATE AFTER EACH TRANSITION — Quote 1")
print("=" * 80)

print("\n[T1] START → analyze_quote")
print(f"  Precondition: quote_data ✓ ({len(q1['line_items'])} line items)")
print(f"  Postcondition: cost_analysis ✓ ({len(result_1['cost_analysis'])} chars)")
print("  Output preview:")
wrap_print(result_1["cost_analysis"][:400])

print(f"\n{'-' * 80}")
print("[T2] analyze_quote → flag_risks")
print(f"  Precondition: cost_analysis ✓")
print(f"  Postcondition: risk_report ✓ ({len(result_1['risk_report'])} chars)")
print("  Output preview:")
wrap_print(result_1["risk_report"][:400])

print(f"\n{'-' * 80}")
print("[T3] flag_risks → draft_summary")
print(f"  Precondition: risk_report ✓")
print(f"  Postcondition: exec_summary ✓ ({len(result_1['exec_summary'])} chars)")
print("  Output preview:")
wrap_print(result_1["exec_summary"][:400])

print(f"\n{'-' * 80}")
print("[T4/T5] draft_summary → approval_decision → ...")
print(f"  Decision: {result_1['decision'].upper()}")
print(f"  Feedback: {result_1['decision_feedback'][:200]}")

## 19. Streaming View — Watch Transitions Fire

In [ ]:
print("STREAMING EXECUTION — Quote 1")
print("=" * 65)

step = 0
for chunk in app.stream(make_initial_state(q1), {"recursion_limit": 20}):
    step += 1
    for node_name, node_output in chunk.items():
        keys = [k for k in node_output if k != "current_node"]
        details = []
        for k in keys:
            v = node_output[k]
            if isinstance(v, str) and len(v) > 50:
                details.append(f"{k}: {len(v)} chars")
            else:
                details.append(f"{k}: {v}")
        print(f"  Step {step}: {node_name:<22} → {', '.join(details)}")

print(f"\nTotal steps: {step}")
if step > 5:
    print("Note: extra steps are negotiation loop iterations.")

## 20. Run on All Three Quotes

In [ ]:
print("PROCESSING ALL VENDOR QUOTES")
print("=" * 70)

all_results = []

for i, quote in enumerate(VENDOR_QUOTES, 1):
    print(f"\n{'=' * 70}")
    print(f"QUOTE {i}/{len(VENDOR_QUOTES)}: {quote['vendor_name']} — ${quote['total_annual']:,}/yr")
    print("=" * 70)

    result = app.invoke(
        make_initial_state(quote),
        {"recursion_limit": 20},
    )
    all_results.append(result)

    print(f"  → {result['decision'].upper()} | Revisions: {result['revision_count']} | "
          f"Report: {len(result['final_report'])} chars")

print(f"\nAll {len(VENDOR_QUOTES)} quotes processed.")

---

# Part E — Cross-Vendor Comparison

## 21. Decision Summary Table

In [ ]:
print("VENDOR COMPARISON")
print("=" * 85)
print(f"{'Vendor':<25} {'Annual':>12} {'Term':>8} {'Decision':>12} {'Revisions':>10} {'Report':>10}")
print("-" * 77)

for r, q in zip(all_results, VENDOR_QUOTES):
    print(f"  {q['vendor_name']:<23} ${q['total_annual']:>10,} {q['contract_term']:>8} "
          f"{r['decision'].upper():>12} {r['revision_count']:>10} {len(r['final_report']):>8} ch")

# Basic stats
approved = [r for r in all_results if r["decision"] == "approve"]
negotiated = [r for r in all_results if r["decision"] == "negotiate"]
rejected = [r for r in all_results if r["decision"] == "reject"]

print(f"\n  Approved:   {len(approved)}")
print(f"  Negotiate:  {len(negotiated)}")
print(f"  Rejected:   {len(rejected)}")

In [ ]:
# Cost comparison
print("COST COMPARISON")
print("=" * 65)

df_cost = pd.DataFrame([
    {
        "Vendor": q["vendor_name"],
        "Annual": q["total_annual"],
        "Total Contract": q["total_contract"],
        "Term (months)": int(q["contract_term"].split()[0]),
        "Monthly": round(q["total_annual"] / 12),
        "Line Items": len(q["line_items"]),
        "Decision": r["decision"].upper(),
    }
    for q, r in zip(VENDOR_QUOTES, all_results)
])

print(df_cost.to_string(index=False))

cheapest = df_cost.loc[df_cost["Annual"].idxmin(), "Vendor"]
most_expensive = df_cost.loc[df_cost["Annual"].idxmax(), "Vendor"]
spread = df_cost["Annual"].max() - df_cost["Annual"].min()
print(f"\n  Annual cost spread: ${spread:,}")
print(f"  Cheapest: {cheapest}")
print(f"  Most expensive: {most_expensive}")

## 22. View Final Reports

In [ ]:
for i, (result, q) in enumerate(zip(all_results, VENDOR_QUOTES), 1):
    print(f"\n{'#' * 80}")
    print(f"# VENDOR {i}: {q['vendor_name']} — {result['decision'].upper()}")
    print(f"{'#' * 80}")
    # Show exec summary + decision (not full report to save space)
    print("\n--- EXECUTIVE SUMMARY ---")
    wrap_print(result["exec_summary"])
    print(f"\n--- DECISION: {result['decision'].upper()} ---")
    wrap_print(result["decision_feedback"][:300])
    print()

---

# Part F — State & Design Analysis

## 23. State Accumulation

In [ ]:
print("STATE ACCUMULATION — Quote 1")
print("=" * 60)

r = all_results[0]
fields = [
    ("cost_analysis", r["cost_analysis"]),
    ("risk_report", r["risk_report"]),
    ("exec_summary", r["exec_summary"]),
    ("decision_feedback", r["decision_feedback"]),
    ("final_report", r["final_report"]),
]

cumulative = 0
for name, content in fields:
    size = len(content)
    cumulative += size
    bar = "█" * min(size // 60, 40)
    print(f"  {name:<22} {size:>5} chars  {bar}")

print(f"\n  Total state: {cumulative:,} chars")
print("  Each field was written by exactly ONE node.")
print("  No field was accidentally overwritten.")

## 24. Transition Summary

```
  TRANSITION MAP (complete)
  ════════════════════════════════════════════════════════════════════

  TRANSITION              FROM → TO               WHAT CHANGES
  ──────────────────────  ─────────────────────── ─────────────────
  T1 (unconditional)      START → analyze_quote   + cost_analysis
  T2 (unconditional)      analyze → flag_risks    + risk_report
  T3 (unconditional)      flag → draft_summary    + exec_summary
  T4 (unconditional)      draft → approval        + decision
                                                  + decision_feedback
                                                  + revision_count
  T5a (conditional: approve)  approval → compile  + final_report
  T5b (conditional: reject)   approval → compile  + final_report
  T5c (conditional: negotiate) approval → draft   (revises exec_summary)
  T6 (unconditional)      compile → END           pipeline done

  LOOP BEHAVIOR:
  When T5c fires, the graph re-enters draft_summary.
  The node sees decision_feedback and incorporates negotiation points.
  Then T4 fires again (draft → approval) with the revised summary.
  This continues until approval, rejection, or MAX_REVISIONS.

  INVARIANTS:
  - cost_analysis is written ONCE and never changes
  - risk_report is written ONCE and never changes
  - exec_summary may be rewritten on negotiation loops
  - revision_count monotonically increases
  - final_report is written exactly once (terminal node)
```

## 25. When to Use Each Pattern

| Pattern | This Pipeline's Use | When to Apply |
|---------|--------------------|--------------|
| **Deterministic node** | analyze_quote (pandas math) | When exact numbers matter |
| **LLM node** | flag_risks, draft_summary | When judgment/synthesis is needed |
| **Conditional edge** | route_after_decision | When outcome determines next step |
| **Revision loop** | negotiate → draft_summary | When iterative refinement improves output |
| **Loop cap** | MAX_REVISIONS = 2 | Always — prevents infinite loops |
| **Partial update** | every node | Always — prevents accidental overwrites |

## 26. Exercises

### Exercise 1: Weighted Scoring
Modify `approval_decision` to weight dimensions differently — for example, compliance_readiness counts 2x for healthcare customers. Pass the weights as part of the initial state.

### Exercise 2: Competitive Comparison Node
Add a `compare_vendors` node after all quotes are processed. It takes the three final reports and produces a ranked recommendation with a winner. You'll need an outer loop or a second graph.

### Exercise 3: Interactive Negotiation
Replace the LLM decision with `input()` — let a human procurement manager type "approve", "negotiate", or "reject" and add negotiation points. Test the revision loop interactively.

### Mini Challenge: Budget Threshold
Add a `budget_check` node that compares the quote's total against a budget ceiling (e.g., \$700K/year). If the quote exceeds the ceiling, automatically route to "negotiate" with feedback requesting a price reduction to fit within budget.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Quotes analyzed:    {len(all_results)}")
print(f"Approved:           {len(approved)}")
print(f"Negotiate:          {len(negotiated)}")
print(f"Rejected:           {len(rejected)}")
print()
print("State transitions:")
print("  T1: START   → analyze_quote     + cost_analysis")
print("  T2: analyze → flag_risks        + risk_report")
print("  T3: flag    → draft_summary     + exec_summary")
print("  T4: draft   → approval_decision + decision, feedback")
print("  T5: approval→ compile_report    (approve/reject)")
print("      approval→ draft_summary     (negotiate loop)")
print("  T6: compile → END               + final_report")
print()
print("Nodes built:")
print("  1. analyze_quote     — deterministic cost benchmarking (no LLM)")
print("  2. flag_risks        — LLM risk identification with mitigations")
print("  3. draft_summary     — LLM executive summary with recommendation")
print("  4. approval_decision — LLM scoring on 5 procurement dimensions")
print("  5. compile_report    — deterministic report assembly")
print()
print("Key patterns:")
print("  • Explicit pre/postconditions for every transition")
print("  • Three-way conditional routing (approve/negotiate/reject)")
print("  • Negotiation loop with revision cap")
print("  • Hybrid pipeline: deterministic math + LLM reasoning")
print("  • Partial state updates — each node returns only its fields")
print("  • Cross-vendor comparison for procurement decision support")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Every edge is a state transition** — define preconditions and postconditions |
| 2 | **Use deterministic nodes for math** — cost analysis should be pandas, not LLM |
| 3 | **Use LLM nodes for judgment** — risk flagging and summary writing need reasoning |
| 4 | **Three-way routing is powerful** — approve/negotiate/reject covers real procurement decisions |
| 5 | **Negotiation loops add value** — revised summaries incorporate counter-points |
| 6 | **Cap all loops** — `MAX_REVISIONS` prevents runaway negotiation |
| 7 | **State fields have OWNERS** — each field is written by exactly one node |
| 8 | **Partial updates prevent overwrites** — nodes return only what they changed |
| 9 | **Benchmark data makes analysis actionable** — "$420/instance" means nothing without "market is $450" |
| 10 | **The graph IS the procurement process** — explicit, testable, and audit-ready |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*